# 10 · Final audit and report tables

Runs the full audit (ASR, over-refusal, task, general capability, with 95% bootstrap CIs) on the **test** splits for the base model, every drifted test-run checkpoint, and every repair from `repair_specs.json`; compiles Tables A–D and the figures; exports a sample for judge validation (docs/05).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · outputs of **01**, **05**, **06**, **07**, **08**, **09** |
| **Reads** | everything above |
| **Writes** (`/kaggle/working/`) | `report/tables.md`, `report/audit.jsonl`, `report/table_c.csv`, `report/judge_validation_sample.csv`, figures |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

Run only **after** the evaluation protocol is frozen (docs/09 Day 9). Test-split numbers computed here go in the report.

In [ ]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

In [ ]:
import json, random
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
JUDGE = "string" if sp.SMOKE else "llamaguard"
OUT = sp.out_dir("report")
sp.hf_login()
bundle, tensors = sp.load_bundle(sp.find_input("probe_bundle.json").parent)
specs = sp.load_json(sp.find_input("repair_specs.json"))
mon = pd.DataFrame(sp.load_jsonl(sp.find_input("monitor.jsonl")))
runs = sp.list_runs()
test_sets = {
    "harm": [x for x in sp.load_jsonl(sp.find_input("eval_harm.jsonl")) if x["split"] == "test"],
    "overrefusal": [x for x in sp.load_jsonl(sp.find_input("eval_overrefusal.jsonl")) if x["split"] == "test"],
    "task": [x for x in sp.load_jsonl(sp.find_input("eval_task.jsonl")) if x["split"] == "test"],
    "general": [x for x in sp.load_jsonl(sp.find_input("eval_general.jsonl")) if x["split"] == "test"],
}
if sp.SMOKE:
    test_sets = {k: v[:4] for k, v in test_sets.items()}
test_runs = sorted(r for r in mon.run.unique() if sp.parse_run(r)[1] != 0 and sp.parse_run(r)[0] != "harm")
if not test_runs:
    print("!! No test-seed runs yet: falling back to DEV runs. These numbers are NOT for the final report.")
    test_runs = sorted(r for r in mon.run.unique() if sp.parse_run(r)[0] != "harm")
print("audited runs:", test_runs, {k: len(v) for k, v in test_sets.items()})

In [ ]:
try:
    judge = sp.Judge(JUDGE)
except Exception as e:
    print(f"!! {JUDGE} unavailable ({type(e).__name__}); using string-match judge")
    judge = sp.Judge("string")
model, tok = sp.load_model(MODEL, dtype=torch.float32)
base_sd = sp.state_dict_cpu(model)
comps, r_star, layer = specs["components"], tensors["directions"][specs["layer"]], specs["layer"]
M = specs["methods"]
audit_path = OUT / "audit.jsonl"
prev = sp.find_input("audit.jsonl", required=False)
done = sp.load_jsonl(prev) if prev else []
if prev and prev != audit_path:
    sp.save_jsonl(done, audit_path)
done_keys = {(r["run"], r["model"]) for r in done}

def record(run, name, changed, handles=()):
    if (run, name) in done_keys:
        return
    with sp.hooked(list(handles)):
        res = sp.audit(model, tok, judge, test_sets)
    row = {"run": run, "model": name, "params_changed": changed, **sp.summary(res),
           "ci95": {k: v["ci95"] for k, v in res.items()},
           "flags": {k: [bool(x) for x in v["flags"]] for k, v in res.items()},
           "responses_sample": {k: v["responses"][:50] for k, v in res.items() if "responses" in v}}
    with audit_path.open("a") as f:
        f.write(json.dumps(row) + "\n")
    done_keys.add((run, name))
    print(run, name, sp.summary(res))

sp.restore_weights(model, base_sd)
record("base", "Base", 0)

In [ ]:
slices = lambda cs: sum((sp.component_slices(model, c) for c in cs), [])
for run in test_runs:
    g = mon[mon.run == run]
    step = int(g.loc[g.asr.idxmax()].step)                 # most-drifted checkpoint, by monitor ASR
    ckpt = dict(runs[run])[step]
    sp.restore_weights(model, base_sd)
    model = sp.apply_adapter(model, ckpt)
    ft_sd = sp.state_dict_cpu(model)

    def fresh():
        sp.restore_weights(model, ft_sd)

    record(run, "Drifted (R0)", 0)
    fresh(); record(run, "R1 circuit restore", sp.interpolate_toward_base(model, base_sd, slices(comps), M["R1"]["alpha"]))
    fresh(); record(run, "R2 circuit projection",
                    sp.project_delta(model, base_sd, sum((sp.writer_slices(model, c) for c in comps), []), r_star, M["R2"]["beta"]))
    fresh(); record(run, "R4 whole-model interpolation",
                    sp.interpolate_toward_base(model, base_sd, sp.all_param_slices(model), M["R4"]["alpha"]))
    for s in M["R5"]["seeds"]:
        fresh()
        rnd = sp.random_components_like(comps, model, random.Random(s))
        record(run, f"R5 random components #{s}", sp.interpolate_toward_base(model, base_sd, slices(rnd), M["R5"]["alpha"]))
    fresh(); record(run, "R6 steering", 0, sp.add_direction(model, layer, r_star, M["R6"]["alpha"]))
sp.restore_weights(model, base_sd)

## Table C — repair (mean ± std over test runs; R5 pooled over draws)

In [ ]:
aud = pd.DataFrame(sp.load_jsonl(audit_path))
aud["method"] = aud.model.str.replace(r" #\d+$", "", regex=True)
metrics = ["asr", "over_refusal", "task", "general"]
agg = aud.groupby("method")[metrics + ["params_changed"]].agg(["mean", "std"])
table_c = pd.DataFrame({m: agg[m]["mean"].map("{:.3f}".format) + " ± " + agg[m]["std"].fillna(0).map("{:.3f}".format)
                        for m in metrics})
table_c["params_changed"] = agg["params_changed"]["mean"].astype(int)
order = ["Base", "Drifted (R0)", "R1 circuit restore", "R2 circuit projection", "R4 whole-model interpolation",
         "R5 random components", "R6 steering"]
table_c = table_c.reindex([o for o in order if o in table_c.index])
table_c.to_csv(OUT / "table_c.csv")
table_c

In [ ]:
# Paired bootstrap over prompts: R1 minus each baseline (negative ASR difference = R1 safer).
comparisons = []
r1 = aud[aud.model == "R1 circuit restore"]
for other in ["Drifted (R0)", "R2 circuit projection", "R4 whole-model interpolation", "R6 steering"] + \
             sorted(aud[aud.method == "R5 random components"].model.unique()):
    o = aud[aud.model == other]
    for metric in metrics:
        a, b = [], []
        for run in r1.run:
            if run in set(o.run):
                a += r1[r1.run == run].flags.iloc[0][metric]
                b += o[o.run == run].flags.iloc[0][metric]
        if a:
            mean, lo, hi = sp.paired_bootstrap_diff(a, b)
            comparisons.append({"vs": other, "metric": metric, "R1_minus_other": mean, "ci95": [lo, hi]})
comp = pd.DataFrame(comparisons)
comp.to_csv(OUT / "paired_comparisons.csv", index=False)
comp[comp.metric == "asr"]

## Tables A, B, D and figures

In [ ]:
table_a = pd.DataFrame(bundle["table_a"])
table_b = pd.read_csv(sp.find_input("table_b.csv"))
table_d = pd.read_csv(sp.find_input("table_d.csv"))
thresholds = sp.load_json(sp.find_input("thresholds.json"))

def to_md(frame, index=False):
    try:
        return frame.to_markdown(index=index)
    except ImportError:
        return frame.to_string(index=index)

text = "\n\n".join([
    "# SafetyPulse results (generated by notebook 10)",
    f"Judge: {judge.name}. Primary CIS: {thresholds['metric']} (tau_fail={thresholds['tau_fail']}), chosen on seed-0 runs.",
    "## Table A — calibration validity", to_md(table_a),
    "## Table B — monitoring", to_md(table_b),
    "## Table C — repair (test splits)", to_md(table_c, index=True),
    "## Paired comparisons (R1 minus other)", to_md(comp),
    "## Table D — cost", to_md(table_d),
])
(OUT / "tables.md").write_text(text, encoding="utf-8")

fig, ax = plt.subplots(figsize=(7, 4))
means = aud.groupby("method")[["asr", "task"]].mean().reindex(table_c.index)
ax.scatter(means.asr, means.task)
for name, r in means.iterrows():
    ax.annotate(name, (r.asr, r.task), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("ASR (lower is safer)"); ax.set_ylabel("task score (GSM8K)"); ax.set_title("Repair: safety vs. utility")
fig.tight_layout(); fig.savefig(OUT / "repair_comparison.png", dpi=150)
print(text[:3000])

## Judge validation (docs/05 §3)

Exports a stratified sample of judged completions. Two people label `human_harmful` (1/0) without looking at
`judge_harmful`; save as `judge_validation_labeled.csv`, attach it, and rerun the next cell for accuracy and
Cohen's κ. If κ < 0.6, switch judge or report the results with that caveat. This file contains harmful
model outputs: keep it private.

In [ ]:
rng = random.Random(0)
items = []
for _, row in aud[aud.model.isin(["Drifted (R0)", "R1 circuit restore", "Base"])].iterrows():
    for i, (resp, flag) in enumerate(zip(row.responses_sample.get("asr", []), row.flags["asr"])):
        items.append({"id": f"{row.run}|{row.model}|{i}", "prompt": test_sets["harm"][i]["prompt"], "response": resp,
                      "judge_harmful": int(flag), "human_harmful": ""})
pos = [x for x in items if x["judge_harmful"]]
neg = [x for x in items if not x["judge_harmful"]]
sample = rng.sample(pos, min(50, len(pos))) + rng.sample(neg, min(50, len(neg)))
rng.shuffle(sample)
sp.save_csv(sample, OUT / "judge_validation_sample.csv")

labeled = sp.find_input("judge_validation_labeled.csv", required=False)
if labeled:
    lab = [r for r in sp.load_csv(labeled) if str(r["human_harmful"]).strip() in ("0", "1")]
    j = np.array([int(r["judge_harmful"]) for r in lab]); h = np.array([int(r["human_harmful"]) for r in lab])
    po = float((j == h).mean()); pe = j.mean() * h.mean() + (1 - j.mean()) * (1 - h.mean())
    print(f"judge vs human on {len(lab)} items: accuracy {po:.2f}, Cohen's kappa {(po - pe) / (1 - pe):.2f}")
else:
    print(f"exported {len(sample)} items for labelling")